# Indonesian ensemble rankings locations

This notebook creates point locations where tidal model rankings will be calculated.

### Load packages

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

In [ ]:
# coastlines 0.0.5
# loading from s3 will take some time
# if the process is to be run multiple times, consider downloading the file first
rates = gpd.read_file("s3://piksel-staging-public-data/ls_coastlines/coastlines_0.0.5.gpkg", layer="rates_of_change")

In [ ]:
rates.head()

In [ ]:
good_rates = rates[rates["certainty"]=="good"]

In [ ]:
good_rates.rate_time.plot.hist(bins=np.arange(-10, 10, 0.2));

In [ ]:
rates.columns

In [ ]:
def thin_random_points(gdf, min_dist=10000, seed=42, keep_all=False):
    # sampling rate calulated from 30 m to min_dist spacing
    if keep_all:
        frac = 1.0
    else:
        frac = np.round(30./min_dist, decimals=5)
    gdf_m = gdf.sample(frac=frac, random_state=seed)
    print(f"Sampled {len(gdf_m)} points from {len(gdf)} total points.")

    kept = []

    for geom in gdf_m.geometry:
        if all(geom.distance(p) >= min_dist for p in kept):
            kept.append(geom)
    print(f"Kept {len(kept)} points after thinning.")

    return gpd.GeoDataFrame(geometry=kept, crs=gdf.crs)

min_dist = 20000
keep_all = False
if keep_all:
    random_sampled_file = f"ensemble_points_sample_{min_dist}.geojson"
else:
    random_sampled_file = f"ensemble_points_random_sample_{min_dist}.geojson"
if os.path.exists(random_sampled_file):
    random_sampled = gpd.read_file(random_sampled_file)
else:
    random_sampled = thin_random_points(rates, min_dist=min_dist, keep_all=keep_all)
    random_sampled.to_file(random_sampled_file)


In [ ]:
random_sampled.explore()